***

Preparing Workspace

***

In [ ]:
import numpy as np
import pandas as pd
import os
from tqdm import tqdm
import re
from datetime import date
import requests
import ast
import xlwt
from xlwt.Workbook import *
from pandas import ExcelWriter
import xlsxwriter
import time
import functools as ft
import urllib.request, json 
pd.set_option('display.max_columns', None)


In [ ]:
# Define user
user = os.getlogin()
path_users = os.path.join('C:\\Users', user)

## Set file paths
if user == 'jfontes':
    # Git
    path_git = os.path.join(path_users, 'Documents', 'Projects', 'Regional-Monitoring', 'Indicator_Gen')

    # SharePoint
    path_out  = os.path.join(path_users
                             , 'Sacramento Area Council of Governments\Regional Monitoring and Reporting - Documents'
                             , 'Process Revamp'
                             , 'Task 9. Collect new data'
                             , 'EPA')

path_code    = os.path.join(path_git, 'Data', 'EPA')
path_config0 = os.path.join(path_git , 'config')
path_config  = os.path.join(path_code, 'config')

In [ ]:
## User defined functions
exec(open(os.path.join(path_config0, 'Functions.py')).read())

***

Importing

***

In [ ]:
print('PM25 data:')
df_pm25 = pd.read_excel(os.path.join(path_out, 'Health_3_MSA_EPA_PM25_raw.xlsx'))
print(df_pm25.shape)
display(df_pm25.head(3))
print('');print('')

print('O3 data:')
df_O3 = pd.read_excel(os.path.join(path_out, 'Health_3_MSA_EPA_O3_raw.xlsx'))
print(df_O3.shape)
display(df_O3.head(3))
print('')

In [ ]:
df_pm25 = df_pm25[~df_pm25['aqi'].isna()]
df_O3   = df_O3  [~df_O3  ['aqi'].isna()]

threshold = 100

aqi_grouping = ['cbsa_code', 'cbsa', 'county', 'state_code', 'county_code', 'site_number', 'date_local', 'local_site_name', 'parameter']

df_pm25 = df_pm25.groupby(aqi_grouping, as_index=False)['aqi'].mean()
df_O3   = df_O3  .groupby(aqi_grouping, as_index=False)['aqi'].mean()

df_pm25['aqi'] = round(df_pm25['aqi'])
df_O3  ['aqi'] = round(df_O3  ['aqi'])

df_aqi = pd.concat([df_pm25, df_O3])

df_aqi['violation'] = 0
df_aqi.loc[df_aqi['aqi'] >= threshold, 'violation'] = 1

aqi_grouping = ['cbsa_code', 'cbsa', 'county', 'state_code', 'county_code', 'site_number', 'date_local', 'local_site_name']

df_aqi = df_aqi.groupby(aqi_grouping, as_index=False)['violation'].mean()

df_aqi = df_aqi[df_aqi['violation'] > 0]
df_aqi = df_aqi.reset_index(drop=True)

print(df_aqi.shape)
display(df_aqi.head())